# 05 · MLflow Experiment Tracking & Model Registry

MLflow gives us reproducibility and auditability:
- Every training run is logged with its hyperparameters, metrics, and model artifact
- The best model is registered in the **Model Registry** under a versioned name
- A `production` alias makes the serving URI stable across model updates

This notebook re-trains all 6 models inside MLflow runs (same code as notebook 03)
and then promotes the best XGBoost to the `production` alias.

> **MLflow 3.x note**: Model *stages* (Staging/Production/Archived) were removed
> in MLflow 3.0 in favour of *aliases* and *tags*.  We use
> `set_registered_model_alias("production")` which is the current API.
> Query the production model with `models:/fraud-detector@production`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from pathlib import Path

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, f1_score, precision_recall_curve, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

DATA_DIR  = Path("../data")
PLOTS_DIR = Path("../plots")
PLOTS_DIR.mkdir(exist_ok=True)

DB_URI    = "sqlite:///" + (Path("..") / "mlflow.db").resolve().as_posix()
EXP_NAME  = "fraud-detection"
REG_NAME  = "fraud-detector"
THRESHOLD = 0.3

mlflow.set_tracking_uri(DB_URI)
mlflow.set_experiment(EXP_NAME)
print(f"Tracking URI : {DB_URI}")
print(f"Experiment   : {EXP_NAME}")


Tracking URI : sqlite:///Q:/ds projects/fraud-detection/mlflow.db
Experiment   : fraud-detection


In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train_resampled.csv")
y_train = pd.read_csv(DATA_DIR / "y_train_resampled.csv").squeeze()
test    = pd.read_csv(DATA_DIR / "test_engineered.csv")
X_test  = test.drop(columns=["Class","Time","Amount"])
y_test  = test["Class"]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


Train: (454902, 34)  |  Test: (56962, 34)


## Pipeline Definitions (same as notebook 03)

Each model is a `Pipeline(StandardScaler + estimator)` -- the logged artifact
is self-contained and can be loaded with one line:
```python
pipeline = mlflow.sklearn.load_model("models:/fraud-detector@production")
prediction = pipeline.predict_proba(X_raw)
```


In [3]:
def build_pipelines():
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=0.1,
                                       solver="lbfgs", random_state=42)),
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=300,
                                           n_jobs=-1, random_state=42)),
        ]),
        "SVM (linear)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", CalibratedClassifierCV(
                LinearSVC(C=0.1, max_iter=2000, random_state=42), cv=3)),
        ]),
        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05,
                                   max_depth=6, eval_metric="logloss",
                                   random_state=42, n_jobs=-1)),
        ]),
        "LightGBM": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    num_leaves=63, random_state=42,
                                    n_jobs=-1, verbose=-1)),
        ]),
        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(hidden_layer_sizes=(128, 64),
                                   max_iter=200, early_stopping=True,
                                   random_state=42)),
        ]),
    }

def compute_metrics(y_true, proba, threshold=THRESHOLD):
    preds = (proba >= threshold).astype(int)
    tp = int(((preds==1) & (y_true==1)).sum())
    fp = int(((preds==1) & (y_true==0)).sum())
    fn = int(((preds==0) & (y_true==1)).sum())
    tn = int(((preds==0) & (y_true==0)).sum())
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall    = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    p_arr, r_arr, _ = precision_recall_curve(y_true, proba)
    return {
        "auc_roc":   roc_auc_score(y_true, proba),
        "pr_auc":    auc(r_arr, p_arr),
        "precision": precision,
        "recall":    recall,
        "f1":        f1_score(y_true, preds, zero_division=0),
        "fpr":       fp/(fp+tn) if (fp+tn) > 0 else 0.0,
    }


## Training Loop with MLflow Logging

For each model we open a `mlflow.start_run()` context and log:
- **Tag**: `model_name`
- **Params**: all hyperparameters from `get_params()` (serialised to strings where needed)
- **Metrics**: auc_roc, pr_auc, precision, recall, f1, fpr
- **Artifact**: the full pipeline (scaler + model) via `mlflow.sklearn.log_model`
- **Artifact**: the PR curve PNG


In [4]:
pipelines = build_pipelines()
results   = []

print(f"{'Model':<22}  {'AUC-ROC':>8}  {'PR-AUC':>7}  {'F1':>7}  {'Run ID'}")
print("-" * 75)

for name, pipeline in pipelines.items():
    with mlflow.start_run(run_name=name) as run:

        pipeline.fit(X_train, y_train)
        proba = pipeline.predict_proba(X_test)[:, 1]
        m     = compute_metrics(y_test.values, proba)

        # Tag
        mlflow.set_tag("model_name", name)

        # Params -- flatten non-serialisable values to strings
        raw_params   = pipeline.named_steps["clf"].get_params()
        clean_params = {
            k: str(v) if not isinstance(v, (int, float, str, bool, type(None))) else v
            for k, v in raw_params.items()
        }
        mlflow.log_params(clean_params)

        # Metrics
        mlflow.log_metrics(m)

        # PR curve artifact
        p_arr, r_arr, thr = precision_recall_curve(y_test, proba)
        idx = min(np.searchsorted(thr, THRESHOLD), len(p_arr)-2)
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(r_arr, p_arr, lw=2, color="#1565C0",
                label=f"PR AUC={m['pr_auc']:.4f}")
        ax.scatter(r_arr[idx], p_arr[idx], s=90, color="#E53935", zorder=5,
                   label=f"t={THRESHOLD}")
        ax.set(xlabel="Recall", ylabel="Precision", xlim=[0,1], ylim=[0,1.05])
        ax.set_title(f"PR Curve -- {name}", fontweight="bold")
        ax.legend(); plt.tight_layout()
        safe = name.lower().replace(" ","_").replace("(","").replace(")","")
        plot_path = PLOTS_DIR / f"pr_curve_{safe}.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight"); plt.close()
        mlflow.log_artifact(str(plot_path), artifact_path="plots")

        # Model artifact (self-contained Pipeline)
        mlflow.sklearn.log_model(pipeline, artifact_path="model")

        results.append({"model": name, "run_id": run.info.run_id, "metrics": m})

    print(f"  {name:<22}  {m['auc_roc']:>8.4f}  {m['pr_auc']:>7.4f}  "
          f"{m['f1']:>7.4f}  {run.info.run_id[:12]}...")


Model                    AUC-ROC   PR-AUC       F1  Run ID
---------------------------------------------------------------------------


2026/04/04 02:29:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 02:29:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Logistic Regression       0.9617   0.7497   0.0888  11b8ae70319b...


2026/04/04 02:31:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/04/04 02:31:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  LightGBM                  0.9786   0.8680   0.8235  e870f46bf2f2...


2026/04/04 02:32:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 02:32:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  MLP                       0.9529   0.8153   0.7018  fe558725728d...


## Results Table

In [5]:
df_results = (
    pd.DataFrame([{"Model": r["model"], **r["metrics"],
                   "Run ID": r["run_id"][:8]+"..."}
                  for r in results])
    .sort_values("auc_roc", ascending=False)
    .reset_index(drop=True)
)
df_results.index += 1
fmt = {c: "{:.4f}".format for c in ["auc_roc","pr_auc","precision","recall","f1","fpr"]}
display(df_results.style.format(fmt)
        .background_gradient(subset=["auc_roc","f1"], cmap="Greens"))


,Model,auc_roc,pr_auc,precision,recall,f1,fpr,Run ID
1,Random Forest,0.9803,0.8784,0.7227,0.8776,0.7926,0.0006,372f78c2...
2,LightGBM,0.9786,0.8680,0.7925,0.8571,0.8235,0.0004,e870f46b...
3,XGBoost,0.9753,0.8654,0.4778,0.8776,0.6187,0.0017,552277e2...
4,SVM (linear),0.9621,0.7269,0.0442,0.8980,0.0842,0.0335,3371b015...
5,Logistic Regression,0.9617,0.7497,0.0467,0.8980,0.0888,0.0316,11b8ae70...
6,MLP,0.9529,0.8153,0.6154,0.8163,0.7018,0.0009,fe558725...


## Register Best XGBoost in Model Registry

We find the XGBoost run with the highest AUC-ROC and register it under the
name `fraud-detector`.  The `production` alias makes it queryable via a
stable URI regardless of version number.


In [6]:
xgb_runs = [r for r in results if r["model"] == "XGBoost"]
best     = max(xgb_runs, key=lambda r: r["metrics"]["auc_roc"])
run_id   = best["run_id"]
auc_roc  = best["metrics"]["auc_roc"]

print(f"Best XGBoost run : {run_id}")
print(f"AUC-ROC          : {auc_roc:.4f}")
print()
print(f"Registering as '{REG_NAME}'...")
mv = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name=REG_NAME)
print(f"Registered version {mv.version}")


Registered model 'fraud-detector' already exists. Creating a new version of this model...
2026/04/04 02:32:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 552277e2d5be40088665cc05a45b78f6 has no artifacts at artifact path 'model', registering model based on models:/m-3e5185bb7cda4deea0e07185be2e5ede instead


Best XGBoost run : 552277e2d5be40088665cc05a45b78f6
AUC-ROC          : 0.9753

Registering as 'fraud-detector'...
Registered version 2


Created version '2' of model 'fraud-detector'.


In [7]:
client = MlflowClient()
client.set_registered_model_alias(REG_NAME, "production", mv.version)
client.set_model_version_tag(REG_NAME, mv.version, "stage", "production")

print(f"Alias 'production' -> version {mv.version}")
print(f"Stable load URI : models:/{REG_NAME}@production")
print()
print("Load in any notebook or script with:")
print(f"  import mlflow.sklearn")
print(f"  mlflow.set_tracking_uri('{DB_URI}')")
print(f"  model = mlflow.sklearn.load_model('models:/{REG_NAME}@production')")


Alias 'production' -> version 2
Stable load URI : models:/fraud-detector@production

Load in any notebook or script with:
  import mlflow.sklearn
  mlflow.set_tracking_uri('sqlite:///Q:/ds projects/fraud-detection/mlflow.db')
  model = mlflow.sklearn.load_model('models:/fraud-detector@production')


## Verify Round-Trip Load

Load the registered model back and confirm it produces the same AUC-ROC.
This validates that the artifact was serialised and stored correctly.


In [8]:
loaded = mlflow.sklearn.load_model(f"models:/{REG_NAME}@production")
proba_verify = loaded.predict_proba(X_test)[:, 1]
auc_verify   = roc_auc_score(y_test, proba_verify)

print(f"Registered model AUC-ROC : {auc_roc:.6f}")
print(f"Re-loaded model AUC-ROC  : {auc_verify:.6f}")
assert abs(auc_roc - auc_verify) < 1e-9, "Round-trip mismatch!"
print("Round-trip check passed.")


Registered model AUC-ROC : 0.975302
Re-loaded model AUC-ROC  : 0.975302
Round-trip check passed.


## MLflow UI

Launch the tracking server to explore runs interactively:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
# Open http://localhost:5000
```

You'll see:
- The `fraud-detection` experiment with 6 runs
- Sortable metrics columns (try sorting by `auc_roc`)
- The `fraud-detector` model in the Model Registry with version 1 aliased to `production`

**Next**: deploy as a FastAPI endpoint and Streamlit dashboard.
